<a href="https://colab.research.google.com/github/Joel-Wang-Math/MIDAS-AI-for-Engineers/blob/main/Copy_of_day04_part2b_abstracts_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4, Part 2b: Featurizing Text with an LLM Pipeline

**MIDAS Summer Academy**

In Part 2a you used the Groq API interactively. Now we'll make a _pipeline_ for making many API calls programmatically, in order to process a volume of data. We'll be **featurizing** some paper abstracts--extracting structured information from unstructured texts.  The abstracts are open-access papers from behavioral and health psychology.

## Setup

We'll setup our Groq calls similar to in the previous notebook. Note that we'll be using the parameter `temperature=0` to make the output as reproducible as possible.

In [1]:
!pip install -q groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [2]:
from getpass import getpass
from groq import Groq
import json, os, textwrap, time
import pandas as pd

client = Groq(api_key=getpass("midas-academy"))
MODEL = "llama-3.3-70b-versatile"
print("Client ready.")

Paste your Groq API key: ··········
Client ready.


## 1. Extracting features from abstracts

Imagine that we want to better understand patterns in the published literature, and we want to extract the following information from a large number of paper abstracts:

| Feature | Definition |
|---|---|
| `sample_size` | how many participants are in a study |
| `population` | target characteristics of the study sample |
| `method` | does the method involve an experiment, testing associations, or something else? |
| `includes_causal_claim` | does the abstract use language that implies one variable causes or changes another? |

## 2. Load the corpus
The corpus (set of text data) contains 30 abstracts; we will start with 5 of them.

In [3]:
DATA_URL = "https://raw.githubusercontent.com/elleobrien/MIDAS_summer_academy_student/main/day04/real_abstracts"

def load_corpus(name):
    local = f"real_abstracts/{name}"
    return pd.read_csv(local if os.path.exists(local) else f"{DATA_URL}/{name}")

abstracts = load_corpus("abstracts.csv")

SELECTED = ["ABS-001", "ABS-002", "ABS-005", "ABS-006", "ABS-023"]
subset = abstracts[abstracts["abstract_id"].isin(SELECTED)].reset_index(drop=True)
print(f"Corpus: {len(abstracts)} abstracts. Working subset: {len(subset)}.")

Corpus: 30 abstracts. Working subset: 5.


## 3. Our first attempt at featurizing
Let's see what happens when we pass in a simple prompt with a single abstract, as a test.

In [5]:
example = subset.set_index("abstract_id").loc["ABS-002", "abstract"]

question = (
    "What is the sample size, population, and method of this study, "
    "and does it conclude that one thing causes another?\n\n" + example
)
resp = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[{"role": "user", "content": question}],
)
# Wrap each line separately so the model's own line breaks are kept
for line in resp.choices[0].message.content.splitlines():
    print(textwrap.fill(line, width=100))

Here's the information you requested:

1. **Sample size**: The sample size is 188 participants.
2. **Population**: The population is not explicitly stated, but it appears to be a general
population of individuals who can participate in nature-based interventions and mindfulness
exercises. The demographics of the participants (e.g., age, sex, socioeconomic status) are not
provided.
3. **Method**: The study used an experimental design with four conditions: nature therapy, nature
only, mindfulness only, and a control condition. Participants were randomly assigned to one of these
conditions.

As for whether the study concludes that one thing causes another, the answer is not explicitly
stated. The study finds a significant association between the combination of nature and mindfulness
and improved positive mood, but it does not claim to have established a causal relationship between
the two. The language used is correlational, suggesting that the combination of nature and
mindfulness is "as

Suppose you need to run 500 abstracts and put the results in a table, one row per abstract. What is the problem with this output?

We can make two changes to how we call the model to help:

1. Ask for JSON with named keys in the prompt.
2. Turn on JSON mode (`response_format={"type": "json_object"}`), which constrains the API to return valid JSON. Without it, models often wrap JSON in markdown code fences or add a preamble, either of which breaks `json.loads`.

## 4. The simplest possible prompt

Version 1 names the keys and the allowed values, and nothing else.

In [6]:
PROMPT_V1 = """Extract the following features from this research abstract.
Return a JSON object with exactly these keys:

- "sample_size": the number of participants
- "population": who the participants were
- "method": "experimental", "correlational", or "other"
- "includes_causal_claim": true or false

Abstract:
{abstract}"""

def extract(abstract_text, prompt_template):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt_template.format(abstract=abstract_text)}],
    )
    return json.loads(resp.choices[0].message.content)

In [7]:
result = extract(example, PROMPT_V1)
result

{'sample_size': 188,
 'population': 'participants',
 'method': 'experimental',
 'includes_causal_claim': True}

### Exercise: code one abstract yourself

Before running the batch, code the first abstract in the subset by hand. Read ABS-001 and fill in `my_codes` with your own values, then compare with the person next to you.

In [8]:
print(textwrap.fill(subset.set_index("abstract_id").loc["ABS-001", "abstract"], width=100))

my_codes = {
    "sample_size": None,
    "population": None,
    "method": None,
    "includes_causal_claim": None,
}

Background and Objectives : Migraine is a common neurological condition that can affect daily life
and well-being. Lifestyle factors such as physical activity, sleep, diet, and stress are often
discussed in relation to migraine, but their role is not always consistent. This study aimed to
examine how selected lifestyle factors are related to migraine frequency and intensity in adults,
with a focus on physical activity and dietary habits. Materials and Methods : A cross-sectional
study was conducted among 300 adults recruited through an online migraine-focused community from 1
January to 28 February 2026. Participants completed a questionnaire addressing migraine history,
frequency and duration of attacks, pain intensity, physical activity, sleep duration, perceived
stress, and dietary habits. Associations were assessed using Spearman correlation and multiple
linear regression analyses. Results : Most participants were female (88%), and 48% reported
physician-diagnosed migraine. Mean mi

## 5. Run the batch

One call per abstract. The pause between calls stays under the free-tier rate limit.

In [9]:
def run_batch(prompt_template, data):
    records = []
    for row in data.itertuples():
        result = extract(row.abstract, prompt_template)
        result["abstract_id"] = row.abstract_id
        records.append(result)
        print(f"{row.abstract_id} done")
        time.sleep(1)
    cols = ["sample_size", "population", "method", "includes_causal_claim"]
    # reindex instead of [cols]: a key the model failed to return shows up as NaN
    # rather than crashing the run
    return pd.DataFrame(records).set_index("abstract_id").reindex(columns=cols)

df_v1 = run_batch(PROMPT_V1, subset)
df_v1

ABS-001 done
ABS-002 done
ABS-005 done
ABS-006 done
ABS-023 done


,sample_size,population,method,includes_causal_claim
abstract_id,,,,
ABS-001,300,adults with migraine,correlational,False
ABS-002,188,participants,experimental,True
ABS-005,98,university students,experimental,True
ABS-006,570,university students at the Faculty of Sports S...,correlational,False
ABS-023,700,adolescents,correlational,True


## 6. Is the output any good?

Every row is filled in and every value has the right type. That does not tell us whether the values are correct.

The most direct check is to read the outputs next to the inputs; Hamel Husain's [evals FAQ](https://hamel.dev/blog/posts/evals-faq/) calls this error analysis and treats it as the first step in evaluating any LLM pipeline. The cell below prints each abstract with the row the model produced. Start with ABS-001: does the model's row match your `my_codes`? Then read the other four and mark which values you agree with.

In [11]:
for row in subset.itertuples():
    print("=" * 100)
    print(row.abstract_id)
    print(textwrap.fill(row.abstract, width=100))
    print()
    print(textwrap.fill(f"MODEL SAYS: {df_v1.loc[row.abstract_id].to_dict()}", width=100))
    print()

ABS-001
Background and Objectives : Migraine is a common neurological condition that can affect daily life
and well-being. Lifestyle factors such as physical activity, sleep, diet, and stress are often
discussed in relation to migraine, but their role is not always consistent. This study aimed to
examine how selected lifestyle factors are related to migraine frequency and intensity in adults,
with a focus on physical activity and dietary habits. Materials and Methods : A cross-sectional
study was conducted among 300 adults recruited through an online migraine-focused community from 1
January to 28 February 2026. Participants completed a questionnaire addressing migraine history,
frequency and duration of attacks, pain intensity, physical activity, sleep duration, perceived
stress, and dietary habits. Associations were assessed using Spearman correlation and multiple
linear regression analyses. Results : Most participants were female (88%), and 48% reported
physician-diagnosed migraine.

### Discussion

Compare notes with the person next to you. Some places to look:

- **ABS-005**: the abstract reports that 167 students were randomized and that the final analytic sample was 98. What did the model report as `sample_size`? Which number should it be?
- **ABS-001**: a cross-sectional questionnaire study. Is that `correlational` or `other`? Also compare its `population` value with the text: were all 300 participants people with migraine?
- **ABS-002**: how informative is its `population` value? The abstract never describes who the 188 participants were. What should the model record in that case?
- **ABS-006**: the conclusion says perceived barriers to physical activity are "a significant factor in predicting nomophobia" and that reducing those barriers could reduce smartphone addiction. Is that a causal claim?
- **ABS-023**: the design is cross-sectional, but the abstract states that screen exposure "led to a decrease" in sleep quality and says it is "necessary to reduce" screen exposure. Did the model set `includes_causal_claim` to true? Should it?

What kind of changes do you think should we make to our prompting strategy to get better results?

## 7. Version 2: write the definitions down
We can try to add more rules (specifications) to our instructions, so the model doesn't make assumptions on our behalf. Including examples of what is or isn't part of a category can also help. Here's another try at prompting:

In [12]:
PROMPT_V2 = """Extract the following features from this research abstract.
Return a JSON object with exactly these keys:

- "sample_size": the total number of participants enrolled or recruited, as an integer.
  If the abstract reports both an initial and a final analytic sample, report the initial one.
  Use null if no number is stated.

- "population": a short phrase describing who the participants were, including any stated
  age group, occupation, or country (for example "university students in Italy").
  Use null if the participants are not described.

- "method": exactly one of:
  "experimental" = participants were randomly assigned to conditions that the researchers controlled;
  "correlational" = variables were measured but not manipulated, including surveys and
  cross-sectional questionnaire studies;
  "other" = anything else, such as reviews, meta-analyses, or qualitative studies.

- "includes_causal_claim": true if the conclusions state that one variable causes, increases,
  reduces, improves, leads to, or otherwise changes another, or recommend changing one variable
  in order to change another. Statements that only report an association, correlation, or link
  are false. The verb "predicts" by itself counts as association, not causation.
  Judge the claim that is made, not whether the study design justifies it.

Abstract:
{abstract}"""

df_v2 = run_batch(PROMPT_V2, subset)
df_v2

ABS-001 done
ABS-002 done
ABS-005 done
ABS-006 done
ABS-023 done


,sample_size,population,method,includes_causal_claim
abstract_id,,,,
ABS-001,300,adults in an online migraine-focused community,correlational,False
ABS-002,188,None,experimental,True
ABS-005,167,university students,experimental,True
ABS-006,570,university students at the Faculty of Sports S...,correlational,True
ABS-023,700,adolescents in a province in the Black Sea reg...,correlational,True


In [13]:
comparison = df_v1.join(df_v2, lsuffix="_v1", rsuffix="_v2")
comparison[["sample_size_v1", "sample_size_v2",
            "method_v1", "method_v2",
            "includes_causal_claim_v1", "includes_causal_claim_v2"]]

,sample_size_v1,sample_size_v2,method_v1,method_v2,includes_causal_claim_v1,includes_causal_claim_v2
abstract_id,,,,,,
ABS-001,300,300,correlational,correlational,False,False
ABS-002,188,188,experimental,experimental,True,True
ABS-005,98,167,experimental,experimental,True,True
ABS-006,570,570,correlational,correlational,False,True
ABS-023,700,700,correlational,correlational,True,True


Check which cells changed. What further changes might you make to the prompting strategy? What kind of errors might you be concerned about arising if we run this on the rest of the corpus?

## 8. Run the pipeline on the rest of the data.
Now, run the pipeline on all 30 abstracts in the corpus. Then, with the people around you, each pick a random (aim for non-overlapping) subset of results to manually inspect. What do you notice? Are there errors you're still seeing? What could be giving rise to those errors? Discuss your findings together.

In [14]:
# The full run takes about a minute for 30 abstracts.
df_full = run_batch(PROMPT_V2, abstracts)
df_full

ABS-001 done
ABS-002 done
ABS-003 done
ABS-004 done
ABS-005 done
ABS-006 done
ABS-007 done
ABS-008 done
ABS-009 done
ABS-010 done
ABS-011 done
ABS-012 done
ABS-013 done
ABS-014 done
ABS-015 done
ABS-016 done
ABS-017 done
ABS-018 done
ABS-019 done
ABS-020 done
ABS-021 done
ABS-022 done
ABS-023 done
ABS-024 done
ABS-025 done
ABS-026 done
ABS-027 done
ABS-028 done
ABS-029 done
ABS-030 done


,sample_size,population,method,includes_causal_claim
abstract_id,,,,
ABS-001,300,adults in an online migraine-focused community,correlational,False
ABS-002,188,None,experimental,True
ABS-003,125,medical students,correlational,False
ABS-004,201,students and teaching/non-teaching staff,correlational,True
ABS-005,167,university students,experimental,True
ABS-006,570,university students at the Faculty of Sports S...,correlational,True
ABS-007,138,None,experimental,True
ABS-008,209,office workers and bus drivers,experimental,True
ABS-009,161,university students,experimental,False


In [15]:
# Print a random sample of 5 results next to their abstracts.
# Re-run this cell to draw a different sample.
text_by_id = abstracts.set_index("abstract_id")["abstract"]

for abstract_id, coded in df_full.sample(5).iterrows():
    print("=" * 100)
    print(abstract_id)
    print(textwrap.fill(text_by_id[abstract_id], width=100))
    print()
    print(textwrap.fill(f"MODEL SAYS: {coded.to_dict()}", width=100))
    print()

ABS-021
Background Adolescence is a sensitive period for psychological, academic, and social development,
and sports participation has been described as a potential protective factor for academic
performance and psychological well-being. However, limited research has examined the combined
influence of sports involvement, sport type, and family background on adolescents' academic and
psychological outcomes. This study aimed to investigate the associations between organized sport
participation, sport type (football vs. judo), psychological well-being, psychosomatic symptoms,
academic performance, and family socioeconomic background among adolescent boys. Methods The sample
consisted of 52 boys aged 11-14 years from a rural school, divided into football players ( n = 13),
judo athletes ( n = 13), non-athletes ( n = 13), and a contextual subgroup of students with special
educational needs (SEN; n = 13), with the latter included for exploratory purposes only. Data
included school-record-bas

One question these features can answer: how often do studies whose designs can only measure associations state causal conclusions? Cross-tabulating `method` with `includes_causal_claim`:

In [16]:
pd.crosstab(df_full["method"], df_full["includes_causal_claim"])

includes_causal_claim,False,True
method,,
correlational,7,12
experimental,1,10


## 9. Discussion: should we publish this?

Suppose you wanted to report the table above as a finding about the health psychology literature. Are you confident enough in the pipeline to publish it? What evidence would satisfy you? What evidence would satisfy a skeptical reviewer?

Here are some practices that are recommended for evaluating LLM pipelines in industrial settings--so they won't all be applicable for every area of research. I highly recommend you look at the [Evals FAQ](https://hamel.dev/blog/posts/evals-faq/) from which these are drawn!

- **Read outputs until you stop finding new failure types.** Record every problem you see in your own words (the FAQ calls this open coding), group the notes into failure categories, and count them. Stop when new outputs stop adding categories; the FAQ suggests starting with about 100 outputs and stopping once 20 consecutive outputs add nothing new.
- **Build a hand-coded validation sample.** Code a random sample of abstracts yourself, before looking at the model's answers, then measure agreement.
- **Spot-check at scale.** Even after validation, sample and read outputs from every large run; a prompt tuned on one corpus can fail on abstracts from another field.
